# Voice Agent with Gradio Interface
This notebook provides a voice agent with a Gradio UI for interactive conversations.

In [ ]:
import requests
import os
import gradio as gr
from utils.voice_agent import transcribe_audio_to_text, generate_text_response, generate_audio_response
from utils.tts import bentoml_ttx_get_audio, save_audio_to_folder

In [ ]:
# API endpoints
FAST_API_BASE = "http://127.0.0.1:8000"
OLLAMA_API_BASE = "http://127.0.0.1:11434"
BENTO_API_BASE = "http://127.0.0.1:3000"

# Prepare data directories
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

audio_dir = f"{data_dir}/audio"
os.makedirs(audio_dir, exist_ok=True)

conversation_dir = f"{audio_dir}/conversations"
os.makedirs(conversation_dir, exist_ok=True)

audio_test_dir = f"{audio_dir}/test"
os.makedirs(audio_test_dir, exist_ok=True)

# Paths
AUDIO_TEST_PATH_DIR = audio_test_dir

In [ ]:
# Network checks
async def run_network_checks() -> str:
    try:
        check_fast_api = requests.get(FAST_API_BASE, timeout=2)
        fast_api_status = f"✅ FastAPI: {check_fast_api.status_code}"
    except:
        fast_api_status = "❌ FastAPI: Not available"
    
    try:
        check_ollama = requests.get(OLLAMA_API_BASE, timeout=2)
        ollama_status = f"✅ Ollama: {check_ollama.status_code}"
    except:
        ollama_status = "❌ Ollama: Not available"
    
    try:
        check_bento_ml = requests.get(BENTO_API_BASE, timeout=2)
        bento_status = f"✅ BentoML: {check_bento_ml.status_code}"
    except:
        bento_status = "❌ BentoML: Not available"
    
    return f"{fast_api_status}\n{ollama_status}\n{bento_status}"

# Run network checks
import asyncio
status = await run_network_checks()
print(status)

## Gradio Voice Agent Interface

In [ ]:
def process_audio(audio_file):
    """
    Process audio input through the voice agent pipeline:
    1. Transcribe audio to text
    2. Generate LLM response
    3. Convert response to audio
    """
    if audio_file is None:
        return "Please upload an audio file.", None, "", ""
    
    try:
        # Step 1: Transcribe audio to text
        transcription_result = transcribe_audio_to_text(audio_file)
        user_text = transcription_result['text']
        
        # Step 2: Generate text response from LLM
        bot_response = generate_text_response(user_prompt=user_text)
        
        # Step 3: Generate audio response
        output_folder = "data/audio/conversations"
        os.makedirs(output_folder, exist_ok=True)
        
        audio_bytes = bentoml_ttx_get_audio(
            bot_response, 
            "en", 
            api_url="http://127.0.0.1:3000/synthesize"
        )
        
        audio_output_path = f"{output_folder}/response.mp3"
        save_audio_to_folder(audio_bytes, folder=output_folder, filename="response.mp3")
        
        return (
            f"✅ Processing complete!",
            audio_output_path,
            user_text,
            bot_response
        )
    
    except Exception as e:
        return f"❌ Error: {str(e)}", None, "", ""


def chat_with_text(user_message, history):
    """
    Text-based chat interface
    """
    if not user_message:
        return history, ""
    
    try:
        # Generate response
        bot_response = generate_text_response(user_prompt=user_message)
        
        # Update history
        history.append((user_message, bot_response))
        
        return history, ""
    
    except Exception as e:
        history.append((user_message, f"Error: {str(e)}"))
        return history, ""

In [ ]:
# Create Gradio interface
with gr.Blocks(title="Voice Agent", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Voice Agent")
    gr.Markdown("Interact with the AI agent using voice or text!")
    
    with gr.Tabs():
        # Voice Interface Tab
        with gr.Tab("🎤 Voice Chat"):
            gr.Markdown("### Upload an audio file to chat with the agent")
            
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="Upload Audio",
                        type="filepath",
                        sources=["upload", "microphone"]
                    )
                    process_btn = gr.Button("Process Audio", variant="primary")
                
                with gr.Column():
                    status_output = gr.Textbox(label="Status", interactive=False)
                    audio_output = gr.Audio(label="Agent Response (Audio)")
            
            with gr.Row():
                with gr.Column():
                    transcription_output = gr.Textbox(
                        label="Your Message (Transcribed)",
                        lines=3,
                        interactive=False
                    )
                
                with gr.Column():
                    response_output = gr.Textbox(
                        label="Agent Response (Text)",
                        lines=3,
                        interactive=False
                    )
            
            process_btn.click(
                fn=process_audio,
                inputs=[audio_input],
                outputs=[status_output, audio_output, transcription_output, response_output]
            )
        
        # Text Interface Tab
        with gr.Tab("💬 Text Chat"):
            gr.Markdown("### Chat with the agent using text")
            
            chatbot = gr.Chatbot(label="Conversation", height=400)
            
            with gr.Row():
                msg = gr.Textbox(
                    label="Your Message",
                    placeholder="Type your message here...",
                    scale=4
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            
            clear_btn = gr.Button("Clear Conversation")
            
            send_btn.click(
                fn=chat_with_text,
                inputs=[msg, chatbot],
                outputs=[chatbot, msg]
            )
            
            msg.submit(
                fn=chat_with_text,
                inputs=[msg, chatbot],
                outputs=[chatbot, msg]
            )
            
            clear_btn.click(fn=lambda: [], outputs=[chatbot])
    
    gr.Markdown("---")
    gr.Markdown("### 📝 Notes")
    gr.Markdown("""
    - **Voice Chat**: Upload an audio file or record using your microphone
    - **Text Chat**: Type messages directly to chat with the agent
    - Make sure all services (Ollama, BentoML) are running before using the interface
    """)

# Launch the interface
demo.launch(share=False, server_name="127.0.0.1", server_port=7860)

## Testing Individual Components
Below are test cells for individual components (optional)

In [ ]:
# Test FastAPI endpoint (optional)
# with open(file=f"{audio_test_dir}/what-is-your-name.m4a", mode="rb") as file:
#     chat_response = requests.post(
#         url=f"{FAST_API_BASE}/chat/",
#         files={'file': file},
#         data={'conversation_id': "1"}
#     )
# print(chat_response.text)

In [ ]:
# Test transcription (optional)
# text = transcribe_audio_to_text("data/audio/test/what-is-your-name.m4a")
# print(text['text'])

In [ ]:
# Test text generation (optional)
# text_response = generate_text_response(user_prompt="What is your name?")
# print(text_response)

In [ ]:
# Test audio generation (optional)
# audio_bytes = bentoml_ttx_get_audio(
#     "Hello, I am your voice assistant!",
#     "en",
#     api_url="http://127.0.0.1:3000/synthesize"
# )
# save_audio_to_folder(audio_bytes, folder="audio", filename="test_audio.mp3")